# k-Nearest Neighbors — Forensic Glass ClassificationWe apply **k-NN** to the **UCI Glass Identification** dataset, classifying glass fragments into 6 categories (window glass, container glass, headlamp glass, etc.) based on their refractive index and chemical composition. This dataset originally arose from criminology — identifying the type of glass from a crime scene fragment.k-NN is a **non-parametric, instance-based** learner — there is no training phase, only memorization. To classify a new point, we find its `k` closest training neighbors (by Euclidean distance) and take a majority vote.**Dataset:** 214 glass samples, 9 chemical features (refractive index + concentrations of Na, Mg, Al, Si, K, Ca, Ba, Fe), 6 glass types. Source: [UCI](https://archive.ics.uci.edu/dataset/42/glass+identification).

In [ ]:
import osimport urllib.requestDATA_DIR = os.path.join(os.path.dirname(os.getcwd()), "data") if os.path.basename(os.getcwd()) == "notebooks" else "data"os.makedirs(DATA_DIR, exist_ok=True)def download_if_needed(url, filename):    """Download a CSV if it doesn't already exist locally."""    path = os.path.join(DATA_DIR, filename)    if not os.path.exists(path):        print(f"Downloading {filename} from {url}")        urllib.request.urlretrieve(url, path)    return pathimport numpy as npimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsfrom sklearn.model_selection import train_test_split, cross_val_scorefrom sklearn.preprocessing import StandardScalerfrom sklearn.neighbors import KNeighborsClassifierfrom sklearn.metrics import accuracy_score, confusion_matrix, classification_reportsns.set_style("whitegrid")np.random.seed(42)

## 1. Load data

In [ ]:
path = download_if_needed(    "https://raw.githubusercontent.com/jbrownlee/Datasets/master/glass.csv",    "glass.csv",)cols = ["RI", "Na", "Mg", "Al", "Si", "K", "Ca", "Ba", "Fe", "Type"]df = pd.read_csv(path, header=None, names=cols)print("Shape:", df.shape)print("\nClass distribution:")print(df["Type"].value_counts().sort_index())df.head()

The 6 glass types are:- **1**: building windows, float-processed- **2**: building windows, non-float- **3**: vehicle windows, float-processed- **5**: containers- **6**: tableware- **7**: headlamps(Class 4 was dropped from the original UCI release.)

## 2. Visual inspection

In [ ]:
# Show pairwise relationships for a few key featuressns.pairplot(    df[["RI", "Na", "Mg", "Al", "Type"]],    hue="Type", height=1.6, plot_kws={"alpha": 0.7, "s": 16},)plt.show()

Some classes overlap heavily in feature space — this is a harder problem than typical toy datasets, which makes it a great test for k-NN.

## 3. Split + scale

In [ ]:
X = df.drop(columns="Type").valuesy = df["Type"].valuesX_tr, X_te, y_tr, y_te = train_test_split(    X, y, test_size=0.25, random_state=42, stratify=y)scaler = StandardScaler()X_tr_s = scaler.fit_transform(X_tr)X_te_s = scaler.transform(X_te)print(f"Train: {X_tr_s.shape}, Test: {X_te_s.shape}")

## 4. Tune k via 5-fold cross-validation

In [ ]:
ks = range(1, 21)mean_scores = []for k in ks:    cv = cross_val_score(        KNeighborsClassifier(n_neighbors=k), X_tr_s, y_tr, cv=5,    )    mean_scores.append(cv.mean())plt.plot(list(ks), mean_scores, marker="o")plt.xlabel("k"); plt.ylabel("CV accuracy")plt.title("Choosing k via 5-fold cross-validation")plt.tight_layout(); plt.show()best_k = list(ks)[int(np.argmax(mean_scores))]print(f"Best k: {best_k}")

## 5. Fit final model and evaluate

In [ ]:
knn = KNeighborsClassifier(n_neighbors=best_k).fit(X_tr_s, y_tr)preds = knn.predict(X_te_s)print(f"Test accuracy: {accuracy_score(y_te, preds):.4f}\n")print(classification_report(y_te, preds))

## 6. Confusion matrix

In [ ]:
cm = confusion_matrix(y_te, preds, labels=knn.classes_)plt.figure(figsize=(7, 5))sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",            xticklabels=knn.classes_, yticklabels=knn.classes_)plt.xlabel("Predicted glass type")plt.ylabel("Actual glass type")plt.title(f"k-NN (k={best_k}) — Confusion Matrix")plt.tight_layout(); plt.show()

## 7. Decision boundary using two key features

In [ ]:
# Use Mg and Al — two of the most discriminative featuresX2 = df[["Mg", "Al"]].valuessc2 = StandardScaler().fit(X2)X2s = sc2.transform(X2)knn2 = KNeighborsClassifier(n_neighbors=best_k).fit(X2s, y)xx, yy = np.meshgrid(    np.linspace(X2s[:, 0].min()-0.5, X2s[:, 0].max()+0.5, 200),    np.linspace(X2s[:, 1].min()-0.5, X2s[:, 1].max()+0.5, 200),)Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)plt.figure(figsize=(8, 6))plt.contourf(xx, yy, Z, alpha=0.3, cmap="viridis")plt.scatter(X2s[:, 0], X2s[:, 1], c=y, edgecolor="k",            cmap="viridis", s=30)plt.xlabel("Mg (scaled)"); plt.ylabel("Al (scaled)")plt.title(f"k-NN decision boundary (k={best_k}, using Mg + Al only)")plt.tight_layout(); plt.show()

## Takeaways- Glass classification is a genuinely hard problem — accuracy in the 70-75% range, much lower than toy datasets.- The class imbalance (some glass types have only 9-13 samples) hurts k-NN, since rare classes have fewer "votes" in the neighborhood.- **Small `k`** (e.g. k=1) overfits — every training point becomes its own region. **Large `k`** oversmooths and starts predicting the majority class.- **Scaling is essential**: refractive index (~1.5) and Si concentration (~70%) are on completely different scales — without scaling, distance is dominated by Si.- Real-world classification often looks more like this than the perfect clusters in textbook datasets.